# PanOrgan 4.1 — Modular GPU Training & Live Dashboard Notebook

Organized into clean, small steps:
1. **Environment & GPU Check**
2. **Output Directory (Google Drive / Kaggle)**
3. **Download Dataset (`theshoaibme/PanOrgan-Dataset`) & Warm-Start Weights (`theshoaibme/PanOrgan-4`)**
4. **Build DataLoaders & Model (`PanOrgan41Model`)**
5. **Live Real-Time Training Loop** *(Batch progress, Speed, ETA, Loss, Val Acc, Val Macro-F1)*
6. **Calibration (`Temperature Scaling` + `Macro-F1 Thresholds`) & Per-Head Test Report**
7. **Generate & Display Clean Graphs** *(Full-scale Training Curves, Per-Head Metrics, Calibration ECE, Confusion Matrices)*

In [1]:
# Step 1: Install dependencies & verify CUDA GPU
!pip -q install "huggingface_hub>=0.25.0,<2.0" timm scikit-learn tqdm pyyaml matplotlib
import os, torch
print(f"CUDA: {torch.cuda.is_available()} | Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
CUDA: False | Device: CPU


In [2]:
# Step 2: Configure persistent output directory (Google Drive or Kaggle Working)
OUT_ROOT = '/content/PanOrgan_4_1'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_ROOT = '/content/drive/MyDrive/PanOrgan_4_1'
except Exception:
    if os.path.exists('/kaggle/working'):
        OUT_ROOT = '/kaggle/working/PanOrgan_4_1'

PLOTS_DIR = os.path.join(OUT_ROOT, 'plots')
os.makedirs(PLOTS_DIR, exist_ok=True)
print(f'Checkpoints & plots will be saved to: {OUT_ROOT}')

OSError: [Errno 30] Read-only file system: '/content'

In [ ]:
# Step 3: Download code bundle, PanOrgan-4 warm-start weights, and PanOrgan-Dataset
from huggingface_hub import snapshot_download, hf_hub_download
import tarfile, shutil

WORK = '/content/work' if os.path.exists('/content') else '/kaggle/working/work'
os.makedirs(WORK, exist_ok=True)

code_tgz = hf_hub_download('theshoaibme/PanOrgan-Dataset', 'code/panorgan_v4_1_code.tar.gz', repo_type='dataset')
with tarfile.open(code_tgz, 'r:gz') as tf:
    tf.extractall(WORK)

V4_CKPT = '/tmp/panorgan_4_ckpt'
os.makedirs(V4_CKPT, exist_ok=True)
for fn in ['best.pt', 'multiscale_heads.pt']:
    shutil.copy2(hf_hub_download('theshoaibme/PanOrgan-4', fn, repo_type='model'), os.path.join(V4_CKPT, fn))

DS_DIR = snapshot_download(
    'theshoaibme/PanOrgan-Dataset',
    repo_type='dataset',
    local_dir='/tmp/panorgan_dataset',
    ignore_patterns=['code/*', 'master_manifest.parquet', 'master_manifest_v4.parquet'],
)
print('All code, weights, and dataset shards ready!')

In [ ]:
# Step 4: Initialize DataLoaders, PanOrgan41Model, Optimizer & EMA
import sys, copy, time, json
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, average_precision_score, balanced_accuracy_score, confusion_matrix, f1_score, roc_auc_score

for p in [f'{WORK}/V2', f'{WORK}/V2/scripts', f'{WORK}/V3/scripts', f'{WORK}/V4/scripts']:
    if p not in sys.path: sys.path.insert(0, p)

from panorgan.config import load_config, seed_everything
from panorgan.data import BalancedModalityBatchSampler, PanOrganShardDataset, ShardCacheManager
from panorgan.evaluation import TemperatureScaler
from panorgan.models import PanOrganNetV2
from PanOrgan_4_fix_heads import COHORT_HEAD_MAP, MultiScaleCohortHead, pool_multiscale
from PanOrgan_4_train import _compute_ece

seed_everything(42)
cfg = load_config(f'{WORK}/V3/configs/v3_panorgan.yaml')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

df = pd.read_parquet('/tmp/panorgan_dataset/master_manifest_v4_1.parquet')
cls_df = df[df['dataset'].isin(COHORT_HEAD_MAP.keys()) & (df['label'] >= 0)].reset_index(drop=True)
train_df = cls_df[cls_df['split'] == 'train'].reset_index(drop=True)
val_df = cls_df[cls_df['split'] == 'val'].reset_index(drop=True)
test_df = cls_df[cls_df['split'] == 'test'].reset_index(drop=True)
print(f'Split sizes -> Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')

mod_map = {k: int(v['id']) for k, v in cfg['modalities'].items()}
cache_mgr = ShardCacheManager('/tmp/panorgan_dataset', '/tmp/panorgan_dataset', max_cache_gb=40.0)
train_ds = PanOrganShardDataset(train_df, cache_mgr, mod_map, target_hw=(256, 256), augment=True)
val_ds = PanOrganShardDataset(val_df, cache_mgr, mod_map, target_hw=(256, 256), augment=False)
test_ds = PanOrganShardDataset(test_df, cache_mgr, mod_map, target_hw=(256, 256), augment=False)

sampler = BalancedModalityBatchSampler(train_df['dataset'].tolist(), batch_size=32, temperature_alpha=0.25)
train_loader = DataLoader(train_ds, batch_sampler=sampler, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=48, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=48, shuffle=False, num_workers=2, pin_memory=True)

class PanOrgan41Model(nn.Module):
    def __init__(self, base, dropout=0.40):
        super().__init__()
        self.base = base
        specs = {hn: nc for _, (hn, nc) in COHORT_HEAD_MAP.items()}
        self.heads = nn.ModuleDict({h: MultiScaleCohortHead(2688, 512, nc, dropout=dropout) for h, nc in specs.items()})
    def forward(self, images, meta_vec, is_3d, prompts):
        out = self.base(images, meta_vec, is_3d, prompts=prompts, return_features=True)
        ms = pool_multiscale(out)
        out['head_logits'] = {h: head(ms) for h, head in self.heads.items()}
        return out

cw_map = {}
for hname, n_cls in set(COHORT_HEAD_MAP.values()):
    cohorts_h = [c for c, (hn, _) in COHORT_HEAD_MAP.items() if hn == hname]
    sub = train_df[train_df['dataset'].isin(cohorts_h)]['label'].values
    sub = sub[(sub >= 0) & (sub < n_cls)]
    counts = np.maximum(np.bincount(sub, minlength=n_cls).astype(np.float32), 1.0)
    w = (counts.sum() / counts) ** 0.5
    cw_map[hname] = torch.tensor(w / w.mean(), dtype=torch.float32, device=device)

base = PanOrganNetV2.from_config(cfg)
st_best = torch.load(f'{V4_CKPT}/best.pt', map_location='cpu', weights_only=False)
base.load_state_dict({k[len('base.'):]: v for k, v in st_best['model_state_dict'].items() if k.startswith('base.')}, strict=False)
model = PanOrgan41Model(base, dropout=0.40).to(device)
ms_ck = torch.load(f'{V4_CKPT}/multiscale_heads.pt', map_location='cpu', weights_only=False)
for hn, sd in ms_ck['multiscale_heads'].items():
    if hn in model.heads:
        model.heads[hn].load_state_dict(sd, strict=False)

ema_model = copy.deepcopy(model).eval()
for p in ema_model.parameters():
    p.requires_grad_(False)

EPOCHS = 5
total_steps = EPOCHS * len(train_loader)
warmup_steps = max(1, int(0.08 * total_steps))
optimizer = torch.optim.AdamW([
    {'params': model.base.parameters(), 'lr': 1.5e-5, 'weight_decay': 0.08},
    {'params': model.heads.parameters(), 'lr': 8.0e-5, 'weight_decay': 0.08},
])
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda s: (s+1)/warmup_steps if s < warmup_steps else max(0.05, 0.5*(1+np.cos(np.pi*(s-warmup_steps)/max(1,total_steps-warmup_steps)))))
scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))
print('Model, Optimizer, and EMA initialized cleanly.')

In [ ]:
# Step 5: Live Real-Time Training Loop (with clean batch logs, speed, ETA & history)
@torch.no_grad()
def eval_split(m, loader, tta=True):
    m.eval()
    by_h = {h: {'logits': [], 'labels': []} for _, (h, _) in COHORT_HEAD_MAP.items()}
    tloss, nloss = 0.0, 0
    for batch in loader:
        imgs, meta, i3d = batch['image'].to(device), batch['meta_vec'].to(device), batch['is_3d'].to(device)
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            o1 = m.base(imgs, meta, i3d, prompts=batch['prompt'], return_features=True)
            ms = 0.5 * (pool_multiscale(o1) + pool_multiscale(m.base(torch.flip(imgs, dims=[-1]), meta, i3d, prompts=batch['prompt'], return_features=True))) if tta else pool_multiscale(o1)
        for i, ds in enumerate(batch['dataset']):
            if ds not in COHORT_HEAD_MAP: continue
            hn, nc = COHORT_HEAD_MAP[ds]
            lbl = int(batch['label'][i].item())
            if lbl < 0 or lbl >= nc: continue
            lg = m.heads[hn](ms[i:i+1].float())
            tloss += float(F.cross_entropy(lg, torch.tensor([lbl], device=device), weight=cw_map[hn], label_smoothing=0.05).item())
            nloss += 1
            by_h[hn]['logits'].append(lg[0].cpu())
            by_h[hn]['labels'].append(lbl)
    ay, ap, f1s = [], [], []
    for hn, d in by_h.items():
        if not d['logits']: continue
        y = np.array(d['labels'])
        p = torch.stack(d['logits']).argmax(dim=-1).numpy()
        ay.extend(y.tolist()); ap.extend(p.tolist())
        f1s.append(float(f1_score(y, p, average='macro', zero_division=0)))
    return {'loss': tloss/max(1,nloss), 'acc': accuracy_score(ay, ap), 'f1': float(np.mean(f1s)), 'by_h': by_h}

best_f1, history = -1.0, []
start_ep, start_batch = 1, 0
RESUME_PATH = f'{OUT_ROOT}/panorgan_4_1_resume.pt'
if os.path.exists(RESUME_PATH):
    ck = torch.load(RESUME_PATH, map_location='cpu', weights_only=False)
    model.load_state_dict(ck['model'])
    ema_model.load_state_dict(ck['ema'])
    optimizer.load_state_dict(ck['opt'])
    scheduler.load_state_dict(ck['sched'])
    best_f1, history = ck.get('best_f1', -1.0), ck.get('history', [])
    start_ep = ck['epoch']
    start_batch = ck.get('batch_idx', 0)
    print(f'Resuming from Epoch {start_ep}, Batch {start_batch} (best_f1={best_f1:.4f})')

rng = np.random.default_rng(42)
n_batches = len(train_loader)
print(f"{'Epoch':>7} | {'Batch':>11} | {'Speed':>9} | {'ETA':>7} | {'TrLoss':>7} | {'ValLoss':>7} | {'ValAcc':>7} | {'ValF1':>7}")
print('-' * 82)

for ep in range(start_ep, EPOCHS + 1):
    t0 = time.time()
    model.train()
    tr_loss, tr_n = 0.0, 0
    for b_idx, batch in enumerate(train_loader, 1):
        if ep == start_ep and b_idx <= start_batch:
            continue
        imgs, meta, i3d, labels = batch['image'].to(device), batch['meta_vec'].to(device), batch['is_3d'].to(device), batch['label'].to(device)
        ds_list = batch['dataset']
        use_mix = rng.random() < 0.25 and imgs.size(0) > 1
        lam = float(rng.beta(0.4, 0.4)) if use_mix else 1.0
        perm = torch.randperm(imgs.size(0), device=device) if use_mix else None
        m_imgs = lam * imgs + (1.0 - lam) * imgs[perm] if use_mix else imgs

        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            out = model(m_imgs, meta, i3d, prompts=batch['prompt'])
            losses = []
            for i, ds in enumerate(ds_list):
                if ds not in COHORT_HEAD_MAP: continue
                hn, nc = COHORT_HEAD_MAP[ds]
                lbl = int(labels[i].item())
                if lbl < 0 or lbl >= nc: continue
                lg = out['head_logits'][hn][i:i+1].float()
                ce = F.cross_entropy(lg, labels[i:i+1], weight=cw_map[hn], label_smoothing=0.05)
                if use_mix and perm is not None and ds_list[int(perm[i])] == ds:
                    lbl2 = int(labels[perm[i]].item())
                    if 0 <= lbl2 < nc:
                        ce = lam * ce + (1.0 - lam) * F.cross_entropy(lg, labels[perm[i]:perm[i]+1], weight=cw_map[hn], label_smoothing=0.05)
                losses.append(ce)
            loss = torch.stack(losses).mean()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        with torch.no_grad():
            for ep_p, mp_p in zip(ema_model.parameters(), model.parameters()):
                ep_p.data.mul_(0.998).add_(mp_p.data, alpha=0.002)
        tr_loss += float(loss.item()); tr_n += 1

        if b_idx % 50 == 0 or b_idx == n_batches:
            el = time.time() - t0
            spd = max(1, b_idx - (start_batch if ep == start_ep else 0)) / max(el, 1e-6)
            eta = (n_batches - b_idx) / max(spd, 1e-6)
            print(f"  {ep:>2}/{EPOCHS:<2} | {b_idx:>5}/{n_batches:<5} | {spd:>5.2f} b/s | {eta:>5.0f}s  | {tr_loss/tr_n:>7.4f} |    -    |    -    |    -   ", flush=True)
        if b_idx % 100 == 0:
            torch.save({'epoch': ep, 'batch_idx': b_idx, 'model': model.state_dict(), 'ema': ema_model.state_dict(), 'opt': optimizer.state_dict(), 'sched': scheduler.state_dict(), 'best_f1': best_f1, 'history': history}, RESUME_PATH)

    start_batch = 0
    vres = eval_split(ema_model, val_loader, tta=False)
    history.append({'epoch': ep, 'train_loss': tr_loss/max(1,tr_n), 'val_loss': vres['loss'], 'val_acc': vres['acc'], 'val_f1': vres['f1']})
    star = ''
    if vres['f1'] > best_f1:
        best_f1 = vres['f1']
        torch.save({'epoch': ep, 'model_state_dict': ema_model.state_dict(), 'val_macro_f1': best_f1}, f'{OUT_ROOT}/panorgan_4_1_best.pt')
        star = ' ★ BEST'
    torch.save({'epoch': ep + 1, 'batch_idx': 0, 'model': model.state_dict(), 'ema': ema_model.state_dict(), 'opt': optimizer.state_dict(), 'sched': scheduler.state_dict(), 'best_f1': best_f1, 'history': history}, RESUME_PATH)
    print(f"  {ep:>2}/{EPOCHS:<2} | {'EPOCH DONE':>11} | {time.time()-t0:>6.0f}s   |   done  | {tr_loss/max(1,tr_n):>7.4f} | {vres['loss']:>7.4f} | {vres['acc']*100:>6.2f}% | {vres['f1']:>7.4f}{star}", flush=True)

In [ ]:
# Step 6: Temperature Scaling, Macro-F1 Threshold Tuning & Final Test Evaluation
ck = torch.load(f'{OUT_ROOT}/panorgan_4_1_best.pt', map_location='cpu', weights_only=False)
ema_model.load_state_dict(ck['model_state_dict'])
vfin = eval_split(ema_model, val_loader, tta=True)
tfin = eval_split(ema_model, test_loader, tta=True)

ay, ap, per_head = [], [], {}
print(f"{'Clinical Head':<18} | {'N':>5} | {'Acc':>7} | {'BalAcc':>7} | {'F1':>6} | {'AUROC':>6} | {'ECE':>6} | {'Thresh':>6}")
print('-' * 82)
for hn, nc in sorted(set(COHORT_HEAD_MAP.values())):
    vd, td = vfin['by_h'].get(hn, {'logits':[]}), tfin['by_h'].get(hn, {'logits':[]})
    if not td['logits']: continue
    tlg, yte = torch.stack(td['logits']), np.array(td['labels'])
    temp = float(TemperatureScaler().fit(torch.stack(vd['logits']), torch.tensor(vd['labels'], dtype=torch.long))) if vd['logits'] else 1.0
    pte = F.softmax(tlg / temp, dim=-1).numpy()
    th = 0.5
    if nc == 2 and vd['logits']:
        vp, vy = F.softmax(torch.stack(vd['logits'])/temp, dim=-1).numpy()[:, 1], np.array(vd['labels'])
        th = float(max(np.linspace(0.30, 0.70, 41), key=lambda t: f1_score(vy, (vp >= t).astype(int), average='macro', zero_division=0)))
    pred = (pte[:, 1] >= th).astype(int) if nc == 2 else pte.argmax(axis=-1)
    acc, bacc = accuracy_score(yte, pred), balanced_accuracy_score(yte, pred)
    mf1, ece = f1_score(yte, pred, average='macro', zero_division=0), _compute_ece(pte, yte)
    auc = roc_auc_score(yte, pte[:, 1]) if nc == 2 and len(np.unique(yte)) > 1 else (roc_auc_score(yte, pte, multi_class='ovr', average='macro') if len(np.unique(yte)) > 1 else 0.5)
    per_head[hn] = {'n': int(len(yte)), 'acc': float(acc), 'bal_acc': float(bacc), 'f1': float(mf1), 'auroc': float(auc), 'ece': float(ece), 'th': th, 'cm': confusion_matrix(yte, pred).tolist()}
    ay.extend(yte.tolist()); ap.extend(pred.tolist())
    print(f"{hn:<18} | {len(yte):>5} | {acc*100:>6.2f}% | {bacc*100:>6.2f}% | {mf1:>6.4f} | {auc:>6.4f} | {ece:>6.4f} | {th:>6.2f}")

overall_acc = float(accuracy_score(ay, ap))
mean_f1 = float(np.mean([v['f1'] for v in per_head.values()]))
print('-' * 82)
print(f"OVERALL WEIGHTED ACCURACY: {overall_acc*100:.2f}% | MEAN HEAD MACRO-F1: {mean_f1:.4f} | N={len(ay):,}")
with open(f'{OUT_ROOT}/panorgan_4_1_report.json', 'w') as f:
    json.dump({'overall_acc': overall_acc, 'mean_f1': mean_f1, 'history': history, 'per_head': per_head}, f, indent=2)

In [ ]:
# Step 7: Generate & Display Clean Full-Scale Graphs (Training, Per-Head, Calibration, Confusion)
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 10, 'axes.grid': True, 'grid.alpha': 0.3})

# 1. Training History
eps = [h['epoch'] for h in history]
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2), dpi=150)
ax[0].plot(eps, [h['train_loss'] for h in history], 'o-', lw=2, label='Train Loss')
ax[0].plot(eps, [h['val_loss'] for h in history], 's--', lw=2, label='Val Loss')
ax[0].set_title('Loss Convergence (Matched Scale)'); ax[0].set_xlabel('Epoch'); ax[0].legend()
ax[1].plot(eps, [h['val_acc']*100 for h in history], 'o-', lw=2, color='#059669', label='Val Acc (%)')
ax[1].plot(eps, [h['val_f1']*100 for h in history], 's-', lw=2, color='#7c3aed', label='Val Macro-F1 (%)')
ax[1].set_ylim(50, 100); ax[1].set_title('Validation Stability (Full Scale)'); ax[1].set_xlabel('Epoch'); ax[1].legend()
plt.tight_layout(); plt.savefig(f'{PLOTS_DIR}/v4_1_train_history.png', dpi=150); plt.show()

# 2. Per-Head Metrics & Calibration ECE
heads = list(per_head.keys())
x = np.arange(len(heads))
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5), dpi=150)
ax[0].bar(x - 0.25, [per_head[h]['acc']*100 for h in heads], 0.25, label='Accuracy (%)', color='#2563eb')
ax[0].bar(x, [per_head[h]['f1']*100 for h in heads], 0.25, label='Macro-F1 (%)', color='#0891b2')
ax[0].bar(x + 0.25, [per_head[h]['auroc']*100 for h in heads], 0.25, label='AUROC (%)', color='#10b981')
ax[0].set_xticks(x); ax[0].set_xticklabels(heads, rotation=30, ha='right'); ax[0].set_ylim(0, 105)
ax[0].set_title('PanOrgan 4.1 Per-Head Test Performance'); ax[0].legend()
ax[1].bar(heads, [per_head[h]['ece'] for h in heads], color='#f59e0b')
ax[1].axhline(0.05, color='#dc2626', ls='--', label='Target ECE < 0.05')
ax[1].set_xticklabels(heads, rotation=30, ha='right'); ax[1].set_title('Calibration Error (ECE)'); ax[1].legend()
plt.tight_layout(); plt.savefig(f'{PLOTS_DIR}/v4_1_per_head_and_ece.png', dpi=150); plt.show()

# 3. Confusion Matrices Grid
n_h = len(heads); cols = 4; rows = (n_h + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(15, 3.8 * rows), dpi=150)
axes = np.atleast_1d(axes).ravel()
for idx, hn in enumerate(heads):
    cm = np.array(per_head[hn]['cm'])
    axes[idx].imshow(cm, cmap='Blues')
    axes[idx].set_title(f"{hn}\nAcc {per_head[hn]['acc']*100:.1f}% | F1 {per_head[hn]['f1']:.3f}")
    for r in range(cm.shape[0]):
        for c in range(cm.shape[1]):
            axes[idx].text(c, r, str(cm[r, c]), ha='center', va='center', color='white' if cm[r, c] > cm.max()/2 else 'black', fontweight='bold')
    axes[idx].set_xlabel('Pred'); axes[idx].set_ylabel('True')
for j in range(len(heads), len(axes)): axes[j].axis('off')
plt.tight_layout(); plt.savefig(f'{PLOTS_DIR}/v4_1_confusion_matrices.png', dpi=150); plt.show()